In [1]:
from pathlib import Path
from post_processing.NN.DNNManager import DNNManager
import pandas as pd
pd.set_option('display.max_columns', None)  # Display all columns
pd.set_option('display.width', 1000)  # Set a larger width to fit the editor window
BAMBOO_SETUP = DNNManager.BAMBOO_SETUP
Z_OUTPUT_eos = Path('/eos/user/a/anunezde/Z_OUTPUT_eos')

2024-09-23 23:06:17.429419: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Welcome to JupyROOT 6.30/02


In [2]:
workdir = Z_OUTPUT_eos / '2022_even_0920' / 'Reco'
sel_name = 'SL_res_2b_x'
models_yml = 'config/NN_tm_custom.yml'
total_inputs = 'input/vars40.txt'
DNNManagerdir =  'NN_Notebook_HM'

In [19]:
from post_processing import References as Refs
import uproot
class DNNManager_focus(DNNManager):

    def load_data(self) -> list[pd.DataFrame]:
        print(f"\nLoading data ...")

        processes_available = Refs._find_processes(self.RESULTSDIR)
        root_files_available = Refs._find_root_files(self.RESULTSDIR)

        if self.total_inputs is not None:
            with open(self.POSTPROCESSING_NN_FOLDER / self.total_inputs) as file:
                branches = [line.strip() for line in file]
            array_extractor = lambda upfile, sel_name: upfile[sel_name].arrays(branches, library="pd")
        else:
            array_extractor = lambda upfile, sel_name: upfile[sel_name].arrays(library="pd")

        df_list = []
        for process in processes_available:
            process_df = pd.DataFrame()
            process_files = [file for file in root_files_available if file.stem in Refs.PROCESSES_FILES[process]]
            for file in process_files:
                upfile = uproot.open(file)
                upfile_df = array_extractor(upfile, self.sel_name)[:1000000]
                upfile_df['File'] = file.stem
                print(f'Number of events read from {file.stem}: {len(upfile_df)}')
                process_df = pd.concat([process_df, upfile_df], ignore_index=True)
            process_df['Process'] = process
            df_list.append(process_df)

        total_df = pd.concat(df_list, ignore_index=True)
        total_df = pd.get_dummies(total_df, columns=['Process'])

        total_df.reset_index(inplace=True)
        total_df.sort_values(by=['event', 'index'], inplace=True)
        total_df.drop(columns='index', inplace=True)

        print(f"Total_df:\n{total_df}")
        
        return total_df

In [20]:
manager = DNNManager_focus(workdir=workdir, sel_name=sel_name, models_yml=models_yml, total_inputs=total_inputs, DNNManagerdir=DNNManagerdir, verbose=1)
manager.set_mode(mode = 'train_eval')
total_df, DNN_models_params = manager.start()

DNN Manager instantiated:
	WORKDIR:/eos/user/a/anunezde/Z_OUTPUT_eos/2022_even_0920/Reco
	RESULTSDIR:/eos/user/a/anunezde/Z_OUTPUT_eos/2022_even_0920/Reco/results
	DNNMANAGERDIR:/eos/user/a/anunezde/Z_OUTPUT_eos/2022_even_0920/Reco/NN_Notebook_HM
	Total inputs file: /afs/cern.ch/user/a/anunezde/bamboodev/hh/Bamboo_setup/src/post_processing/NN/input/vars40.txt
	Test models file: /afs/cern.ch/user/a/anunezde/bamboodev/hh/Bamboo_setup/src/post_processing/NN/config/NN_tm_custom.yml

Getting test models from: NN_tm_custom.yml

Loading data ...
Number of events read from DY_dl_mll_10to50: 248
Number of events read from DY_dl_mll_50_0J: 65
Number of events read from DY_dl_mll_50_1J: 883
Number of events read from DY_dl_mll_50_2J: 28461
Number of events read from bbWW_dl: 2994
Number of events read from bbWW_sl: 5702
Number of events read from bbtautau: 22128
Number of events read from WW: 1148
Number of events read from WZ: 1570
Number of events read from ZZ: 159
Number of events read from Wj

In [97]:
print(f"Total number in total_df: {total_df.shape}")

Total number in total_df: (1776804, 50)


In [98]:
print(f"\nRun value counts")
print(f"{total_df['run'].value_counts()}")
print("\nLuminosity Block value counts")
print(f"{total_df['luminosityBlock'].value_counts()}")

print("\nbunchCrossing value counts")
print(f"{total_df['bunchCrossing'].value_counts()}")


Run value counts
1    1776804
Name: run, dtype: int64

Luminosity Block value counts
385       222
4115      218
261       213
2331      209
610       206
         ... 
58836       1
66258       1
66262       1
66263       1
144425      1
Name: luminosityBlock, Length: 52697, dtype: int64

bunchCrossing value counts
4294967295    1776804
Name: bunchCrossing, dtype: int64


# Exact Duplicates

In [74]:
pd.reset_option('display.max_rows')
exact_duplicates_bool_total = total_df.duplicated(keep=False)
exact_duplicates = total_df[exact_duplicates_bool_total]
print(f"Number of exactly identical rows: {exact_duplicates.shape[0]}")
print(exact_duplicates)

Number of exactly identical rows: 6224
            event  run  luminosityBlock     genWeight  bunchCrossing  genTtbarId     lep0_pt  lep0_eta  lep0_phi  ak4_jet0_pt  ak4_jet0_eta  ak4_jet0_phi  ak4_jet1_pt  ak4_jet1_eta  ak4_jet1_phi  ak4_jet2_pt  ak4_jet2_eta  ak4_jet2_phi  ak4_btag0_pt  ak4_btag0_eta  ak4_btag0_phi  ak4_btag1_pt  ak4_btag1_eta  ak4_btag1_phi     met_pt   met_phi   bjets_mbb  bjets_dPhi  bjets_dR  bjets_pt_bb  trijet_mInv   trijet_pt  trijet_pt_rat     blnu_mT     blnu_pt  trijet_bijet_dR  trijet_bijet_dPhi  bjet_bijet_dR  bjet_bijet_dPhi     all_pt    all_mInv  all_jets_HT             File  Process_DY  Process_HH_bbWW  Process_HH_bbtautau  Process_VV  Process_WJets  Process_tW  Process_ttbar
434141      29693    1               42      3.805950     4294967295         151  216.248413  0.700439  0.684448     52.09375      0.398804     -2.054199    29.109375      0.391663     -2.911621     27.96875     -1.154785      3.085938     29.109375       0.391663      -2.911621 

In [73]:
print(exact_duplicates['File'].value_counts())

tbarWplus_dl       3530
tbarWplus_sl       2646
DY_dl_mll_50_1J      26
WZ                   16
DY_dl_mll_50_0J       6
Name: File, dtype: int64


# Drop duplicates

In [76]:
total_df_no_dup = total_df.drop_duplicates(keep='first')

# Duplicates in 'event', 'run', 'luminosityBlock', 'bunchCrossing'

In [107]:
print(f"Total number of loaded events with NO duplicates: {total_df_no_dup.shape}")

semidup_bool_total = total_df_no_dup.duplicated(subset=['event', 'luminosityBlock', 'File', 'genTtbarId'], keep=False)
semidup_bool = total_df_no_dup[semidup_bool_total]
proc_columns = [col for col in total_df_no_dup.columns if col.startswith('Process_')]
rel_columns = ['event', 'luminosityBlock', 'genWeight', 'genTtbarId'] + proc_columns + ['File']
print(semidup_bool[rel_columns])

Total number of loaded events with NO duplicates: (1773692, 50)
         event  luminosityBlock  genWeight  genTtbarId  Process_DY  Process_HH_bbWW  Process_HH_bbtautau  Process_VV  Process_WJets  Process_tW  Process_ttbar          File
164801    9636                7  35.987400       10151           0                0                    0           0              0           1              0    tWminus_sl
319430    9636                7  35.987400       10151           0                0                    0           0              0           1              0    tWminus_sl
160053   14767               10  35.987400         151           0                0                    0           0              0           1              0    tWminus_sl
376269   14767               10  35.987400         151           0                0                    0           0              0           1              0    tWminus_sl
526068   15319               11  36.054901         151           0     

In [108]:
semidup_bool[[col for col in total_df.columns if col not in rel_columns]]

,run,bunchCrossing,lep0_pt,lep0_eta,lep0_phi,ak4_jet0_pt,ak4_jet0_eta,ak4_jet0_phi,ak4_jet1_pt,ak4_jet1_eta,ak4_jet1_phi,ak4_jet2_pt,ak4_jet2_eta,ak4_jet2_phi,ak4_btag0_pt,ak4_btag0_eta,ak4_btag0_phi,ak4_btag1_pt,ak4_btag1_eta,ak4_btag1_phi,met_pt,met_phi,bjets_mbb,bjets_dPhi,bjets_dR,bjets_pt_bb,trijet_mInv,trijet_pt,trijet_pt_rat,blnu_mT,blnu_pt,trijet_bijet_dR,trijet_bijet_dPhi,bjet_bijet_dR,bjet_bijet_dPhi,all_pt,all_mInv,all_jets_HT
164801,1,4294967295,31.722099,-1.060547,-1.369370,119.87500,-1.538330,3.125977,103.87500,-1.571289,0.985962,66.81250,-1.519775,2.372070,119.87500,-1.538330,3.125977,103.875000,-1.571289,0.985962,164.011307,-0.475769,198.057007,2.140015,2.140269,108.339333,138.279388,197.459473,0.844633,385.917084,210.402252,0.550229,-0.550056,1.078862,-1.037340,64.487770,723.507507,365.890625
319430,1,4294967295,81.096710,0.087046,1.752472,1438.00000,0.866699,-0.080109,1024.00000,0.627441,3.104980,302.00000,0.421875,2.073730,1024.00000,0.627441,3.104980,302.000000,0.421875,2.073730,107.522514,2.134766,565.641296,1.031250,1.051539,1207.282227,1091.445923,1474.782715,0.914346,1242.556030,1115.035889,0.053329,-0.000235,2.447144,-0.010802,21.092043,3734.452637,3218.656250
160053,1,4294967295,247.741913,-0.079391,3.005859,500.25000,-0.131531,0.008883,71.31250,0.125519,1.433594,50.18750,1.962158,-2.192383,500.25000,-0.131531,0.008883,71.312500,0.125519,1.433594,189.155640,-2.965332,252.800354,1.424711,1.447714,515.481750,633.303162,436.509888,0.728541,509.634583,427.866150,2.361431,-2.200069,2.596302,-2.358542,11.376465,1279.622559,670.468750
376269,1,4294967295,59.195469,1.611313,-1.096695,48.03125,-1.123779,1.723633,44.96875,0.103760,0.669067,37.62500,-1.442627,-2.373535,37.62500,-1.442627,-2.373535,26.703125,0.885986,2.543945,18.719130,-2.067871,100.546410,-1.365705,2.699553,50.377853,134.261826,90.580757,0.756712,248.164825,95.680534,0.440777,-0.290240,2.072058,-1.328423,5.760934,421.395996,157.328125
526068,1,4294967295,51.477802,0.471909,1.967285,115.75000,1.312256,-0.499817,71.37500,0.741089,-1.748535,63.93750,1.219971,2.157227,71.37500,0.741089,-1.748535,63.937500,1.219971,2.157227,57.913853,0.542480,131.651276,2.377424,2.425174,50.921814,278.828033,119.064880,0.521642,197.802673,130.760620,0.620349,0.601647,1.471539,1.235096,52.413723,498.755737,292.187500
644158,1,4294967295,34.553761,-2.407227,1.385498,273.50000,-1.160400,-3.075684,186.00000,-1.920898,0.333496,83.12500,-1.719727,-0.747803,273.50000,-1.160400,-3.075684,83.125000,-1.719727,-0.747803,80.009361,-1.293945,292.581787,2.327881,2.394133,224.685120,173.136688,252.341644,0.832038,476.533813,252.163910,0.324087,0.312450,1.223644,1.203061,25.612652,844.626404,576.781250
514681,1,4294967295,16.513563,-2.354950,-0.668617,176.50000,-2.257812,-2.410645,174.12500,1.004395,-0.606201,143.37500,-1.741455,1.500244,176.50000,-2.257812,-2.410645,108.125000,1.427734,-1.017578,122.047356,0.699341,185.436646,-1.117658,1.541089,222.890259,182.474487,282.601959,0.874083,430.503967,164.876465,0.238288,-0.234326,1.159065,-0.975476,30.393862,2190.466553,808.453125
631428,1,4294967295,119.150703,0.966103,0.502029,153.75000,0.697998,2.125488,117.50000,-0.804688,-1.241211,106.00000,-1.360107,-2.362305,153.75000,0.697998,2.125488,117.500000,-0.804688,-1.241211,52.992992,1.655762,348.164734,2.916486,3.280847,47.176720,160.482605,218.922974,0.864667,344.286011,241.581223,0.479064,-0.474250,1.095924,-1.017617,52.750637,922.129822,491.437500
165024,1,4294967295,17.630180,-1.193314,0.589592,132.00000,-0.413879,0.200378,76.18750,-1.010986,2.161621,49.15625,1.703125,-2.851562,76.18750,-1.010986,2.161621,25.593750,-1.756104,-1.039307,105.689621,-2.409180,95.718529,3.082258,3.171042,50.661533,346.966797,95.734795,0.463046,228.093445,106.400398,0.818075,0.250475,2.821630,1.186736,13.403193,605.456970,282.937500
301663,1,4294967295,62.457027,1.097061,-0.848480,170.12500,0.123184,-2.757812,103.31250,-1.144531,1.421143,101.93750,-2.321289,0.781128,170.12500,0.123184,-2.757812,59.250000

In [109]:
semidup_bool['File'].value_counts()

tbarWplus_sl    20
tWminus_sl       6
Name: File, dtype: int64